In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
housing_full = pd.read_csv(Path("datasets/housing/housing.csv"))

In [ ]:
housing_full.head()

In [ ]:
housing_full.info()

In [ ]:
housing_full.hist(bins=50, figsize=(12, 8))
plt.show()

In [ ]:
housing_full["income_cat"] = pd.cut(housing_full["median_income"],
                                    bins=[0,1.5,3.0,4.5,6, np.inf],
                                    labels=[1,2,3,4,5])

cat_counts = housing_full['income_cat'].value_counts().sort_index()
cat_counts.plot.bar(rot=0, grid = True)
plt.xlabel("Income category")
plt.ylabel("No of districts")
plt.show()
print(cat_counts[1]*0.2)
print(cat_counts[5]*0.2)

In [ ]:
from sklearn.model_selection import train_test_split
strat_train_set, strat_test_set = train_test_split(housing_full, stratify=housing_full["income_cat"], test_size=0.2, random_state=42)
print(len(strat_train_set), len(strat_test_set))

In [ ]:
strat_test_set["income_cat"].value_counts()

In [ ]:
# As we don't need it cause we made seperations already.
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [ ]:
housing = strat_train_set.copy()
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True,
            s=housing["population"] / 100, label="population",
            c="median_house_value", cmap="inferno", colorbar=True,
            legend=True, sharex=False, figsize=(10,7))
plt.show()

In [ ]:
# Finding correlation values/score of each attribute with median_house_value.
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
#Now cleaning process starts to prepare data for ml aogorithms.

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

#Seperated the labels and instances.


Data cleaning data usually means making data reliable and consumable by ml algorithms.
1. Handling missing values.
2. Removing Duplicates.
3. Fix Incorrect data.
4. Handle outliers.
5. Standardize formats like fonts, styles, upper/lowercase, making similar to avoid confusion.
6. Feature Engineering accumlating data two or more attributtes to make more reliable and effective attribute/s. Reduces data complexity and unwanted features.
7. Convert Categorical Data. ml needs numaerical data, so attributes are encoded into numbers or can be categotised.

In [ ]:
# Custom log transformer for scaling
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp, feature_names_out="one-to-one")

In [ ]:
# Custom transformer class for finding clusters and similarity score with each cluster.
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_= KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self
    
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
# Transformation pipelines.
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import  SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import root_mean_squared_error

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
category_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

In [ ]:
# Now for ratios

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ['ratio']

ratio_pipeline = make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(column_ratio, feature_names_out=ratio_name), StandardScaler())

In [ ]:
#For log 
log_pipeline = make_pipeline(SimpleImputer(strategy="median"), log_transformer)

#Kmeans clusers and rbf similarity.
cluster_simil = ClusterSimilarity(n_clusters=45, gamma=1.0, random_state=42) # Best hyperparameter found after tuning.

In [ ]:
#Now we will automate all these under single tranformer:
from sklearn.compose import ColumnTransformer, make_column_selector

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline, ['total_bedrooms', 'total_rooms']),
    ("rooms_per_house", ratio_pipeline, ['total_rooms', 'households']),
    ("people_per_house", ratio_pipeline, ['population', 'households']),
    ("log", log_pipeline, ['total_bedrooms', 'total_rooms', 'population', 'households', 'median_income']),
    ("geo", cluster_simil, ['latitude', 'longitude']),
    ("cat", category_pipeline, make_column_selector(dtype_include=object))
], remainder=num_pipeline)

In [ ]:
# We will be using RandomForestRegressor.
from sklearn.ensemble import RandomForestRegressor

Forest_reg = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42, max_features=9))
])
Forest_reg.fit(housing, housing_labels)

In [ ]:
#Evaluate on trraining set
trained_housing_predictions = Forest_reg.predict(housing)
trained_housing_predictions_rmse = root_mean_squared_error(housing_labels, trained_housing_predictions)
trained_housing_predictions_rmse

In [ ]:
#Cross_Validation
from sklearn.model_selection import cross_val_score

cv_rmse_score = - cross_val_score(Forest_reg, housing, housing_labels, cv=5, scoring="neg_root_mean_squared_error")
pd.Series(cv_rmse_score).describe()

In [ ]:
# Evaluation on test_set.
test_housing = strat_test_set.drop("median_house_value", axis=1)
test_housing_labels = strat_test_set["median_house_value"].copy()

In [ ]:
forest_test_predictions = Forest_reg.predict(test_housing)
forest_test_predictions_rmse = root_mean_squared_error(test_housing_labels, forest_test_predictions)
forest_test_predictions_rmse

In [ ]:
#import joblib

#joblib.dump(Forest_reg, "housing_prediction_model.pkl")

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.value_counts()